# Multimodal Teacher Guided DINOv3 Adapter

Small-scale validation on `zh0416/pv-gf2`: data audit, InternVL semantic cache, frozen DINOv3 SAT feature cache, and controlled A/B segmentation experiments.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle before running.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN loaded from Kaggle Secrets')
except Exception:
    print('HF_TOKEN secret not available; public or mounted model paths will be used')

os.environ['HF_HOME'] = '/kaggle/temp/model_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/kaggle/temp/model_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/temp/model_cache'


In [ ]:
repo = Path('/kaggle/working/DINOV3')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Zh0416/DINOV3.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo), '--no-deps', '--no-build-isolation'], check=True)
config = str(repo / 'configs' / 'kaggle_small.yaml')


## Stage 1 - Dataset inspection and fixed stratified split

In [ ]:
subprocess.run([sys.executable, str(repo / 'scripts' / '01_inspect_data.py'), '--config', config], check=True)

## Stages 3-4 - InternVL semantic JSON and Gaussian attention source

In [ ]:
subprocess.run([sys.executable, str(repo / 'scripts' / '02_generate_teacher.py'), '--config', config], check=True)

## Stage 2 - Frozen DINOv3 ViT-L/16 SAT feature cache

In [ ]:
subprocess.run([sys.executable, str(repo / 'scripts' / '03_cache_dino_features.py'), '--config', config], check=True)

## Stages 5-7 - Baseline and PV-aware adapter experiments

In [ ]:
subprocess.run([sys.executable, str(repo / 'scripts' / '04_train_experiments.py'), '--config', config], check=True)
subprocess.run([sys.executable, str(repo / 'scripts' / '05_compare_experiments.py'), '--config', config], check=True)

In [ ]:
import json
comparison_path = Path('/kaggle/working/outputs/reports/comparison.json')
comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
print(json.dumps(comparison, ensure_ascii=False, indent=2))